# Notebook Thử Nghiệm: Khảo Sát & Ghép Bảng Dữ Liệu Bronze (Listings + Details)

Notebook này dùng để thực hiện **từng bước khảo sát dữ liệu thực tế** trước khi làm sạch và chuẩn hóa sang tầng Silver:

1.  **Nạp Dữ Liệu Master Bronze**: Đọc 2 bảng `bronze_listings_master.json` và `bronze_details_valid.json`.
2.  **Ghép Bảng (Join Listings + Details)**: Kết hợp các trường từ trang danh mục (Logo, tags ngoài) và trang chi tiết (JD sâu, chế độ, yêu cầu).
3.  **Thống Kê Chi Tiết**: Phân tích cấu trúc (`info`), giá trị rỗng/khuyết (`null`/chuỗi rỗng), và trùng lặp (`duplicates`).
4.  **Xem Mẫu Dữ Liệu (Sample 10 Dòng)**: Khảo sát trực quan 10 bản ghi sau khi ghép.

In [5]:
import os
import json
import pandas as pd
import numpy as np
from IPython.display import display

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print(' Môi trường phân tích dữ liệu đã sẵn sàng!')

 Môi trường phân tích dữ liệu đã sẵn sàng!


## 1. Nạp Dữ Liệu Master Từ Tầng Bronze

Đọc 2 tệp tích lũy chuẩn từ `data/bronze/`:
- `bronze_listings_master.json`: Bảng danh mục tin tuyển dụng (kèm tags, logo, trạng thái crawl).
- `bronze_details_valid.json`: Bảng chi tiết việc làm hợp lệ sau khi qua bộ kiểm định Data Quality.

In [6]:
# Duong dan toi 2 file Bronze Master (tuong thich moi thu muc lam viec)
def resolve_path(rel_subpath):
    candidates = [
        rel_subpath,
        f"../{rel_subpath}",
        f"../../{rel_subpath}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return os.path.abspath(rel_subpath)

listings_path = resolve_path('data/bronze/bronze_listings_master.json')
details_path = resolve_path('data/bronze/bronze_details_valid.json')

with open(listings_path, 'r', encoding='utf-8') as f:
    listings_data = json.load(f)
with open(details_path, 'r', encoding='utf-8') as f:
    details_data = json.load(f)

df_listings = pd.DataFrame(listings_data)
df_details = pd.DataFrame(details_data)

print(f'Dang doc Listings tu: {listings_path}')
print(f'Dang doc Details tu : {details_path}\n')
print(f'Bang Listings: {len(df_listings):,} dong, {df_listings.shape[1]} cot')
print(f'Bang Details : {len(df_details):,} dong, {df_details.shape[1]} cot\n')

# Thong ke so luong tin theo tung san
summary_sources = pd.DataFrame({
    'So Listings': df_listings['source'].value_counts(),
    'So Details Valid': df_details['source'].value_counts()
})
display(summary_sources)


Dang doc Listings tu: c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_listings_master.json
Dang doc Details tu : c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_details_valid.json

Bang Listings: 3,809 dong, 20 cot
Bang Details : 3,765 dong, 35 cot



,So Listings,So Details Valid
source,,
topcv,2241,2197
vietnamworks,1421,1421
topdev,147,147


## 2. Ghép 2 Bảng (Join Listings + Details)

Ghép theo cặp khóa định danh duy nhất: `['source', 'source_job_id']`.
- **Mục tiêu**: Bổ sung các cột giá trị từ Listings mà Details có thể chưa có (như `company_logo_url`, `skills_raw`, `posted_at_raw`, `raw_file_name` của listing).
- Giữ nguyên toàn bộ các trường nội dung chi tiết sâu (`overview_raw`, `responsibilities_raw`, `requirements_raw`, `benefits_raw`, `work_mode_raw`, `company_size_raw`...) từ Details.

In [7]:
# Chọn các cột bổ sung từ Listings (tránh xung đột trùng lặp cột)
listing_extra_cols = ['source', 'source_job_id', 'company_logo_url', 'skills_raw', 'posted_at_raw', 'raw_file_name']
df_list_sub = df_listings[listing_extra_cols].drop_duplicates(subset=['source', 'source_job_id']).rename(
    columns={'raw_file_name': 'listing_file_name'}
)

# Ghép bảng Details với Listings theo ['source', 'source_job_id']
df_merged = pd.merge(
    df_details,
    df_list_sub,
    on=['source', 'source_job_id'],
    how='inner'
)

print(f' Kết quả ghép bảng (Join):')
print(f'  • Tổng số dòng sau khi ghép : {len(df_merged):,} dòng')
print(f'  • Tổng số cột sau khi ghép  : {df_merged.shape[1]} cột')
print(f'  • Tỷ lệ khớp khóa (Match)  : {len(df_merged)/len(df_details)*100:.2f}% (Khớp hoàn toàn 100%)')

 Kết quả ghép bảng (Join):
  • Tổng số dòng sau khi ghép : 3,765 dòng
  • Tổng số cột sau khi ghép  : 39 cột
  • Tỷ lệ khớp khóa (Match)  : 100.00% (Khớp hoàn toàn 100%)


## 3. Thống Kê Chi Tiết: Info, Missing Values (Null / Rỗng), và Duplicates (Dup)

Khảo sát toàn diện chất lượng dữ liệu của bảng sau khi ghép:
1. **Cấu trúc & Kiểu dữ liệu (`.info()`)**.
2. **Thống kê giá trị khuyết (`null` và chuỗi rỗng `""`)**: Tính số lượng và tỷ lệ % khuyết trên từng trường dữ liệu.
3. **Kiểm tra trùng lặp (`duplicates`)**: Kiểm tra trùng theo ID nguồn, theo URL, và theo cặp chức danh - công ty.

In [8]:
# 1. Hiển thị thông tin tổng quan kiểu dữ liệu
print('--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---')
df_merged.info(verbose=True, show_counts=True)

# 2. Thống kê Missing Values (Null / None và Chuỗi Rỗng "")
null_counts = df_merged.isnull().sum()
empty_counts = (df_merged == '').sum()
total_missing = null_counts + empty_counts
missing_pct = (total_missing / len(df_merged) * 100).round(2)

df_missing_report = pd.DataFrame({
    'Kiểu Dữ Liệu': df_merged.dtypes.astype(str),
    'Số lượng Null/None': null_counts,
    'Số lượng Chuỗi Rỗng ("")': empty_counts,
    'Tổng Số Khuyết': total_missing,
    'Tỷ Lệ Khuyết (%)': missing_pct
}).sort_values(by='Tỷ Lệ Khuyết (%)', ascending=False)

print('\n--- 2. BẢNG THỐNG KÊ GIÁ TRỊ KHUYẾT (MISSING / NULL REPORT) ---')
display(df_missing_report)

# 3. Kiểm tra Trùng lặp (Duplicates Report)
dup_source_id = df_merged.duplicated(subset=['source', 'source_job_id']).sum()
dup_job_url = df_merged.duplicated(subset=['job_url']).sum()
dup_title_company = df_merged.duplicated(subset=['title_raw', 'company_name_raw']).sum()

print('--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---')
df_dup_report = pd.DataFrame({
    'Tiêu Chí Trùng Lặp': [
        'Theo cặp khóa nguồn [source, source_job_id]',
        'Theo đường dẫn [job_url]',
        'Theo cùng vị trí & cùng công ty [title_raw, company_name_raw]'
    ],
    'Số Bản Ghi Trùng': [dup_source_id, dup_job_url, dup_title_company],
    'Tỷ Lệ Trùng (%)': [
        round(dup_source_id / len(df_merged) * 100, 2),
        round(dup_job_url / len(df_merged) * 100, 2),
        round(dup_title_company / len(df_merged) * 100, 2)
    ]
})
display(df_dup_report)

--- 1. THÔNG TIN CẤU TRÚC (DF INFO) ---
<class 'pandas.DataFrame'>
RangeIndex: 3765 entries, 0 to 3764
Data columns (total 39 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   detail_id             3765 non-null   str   
 1   source                3765 non-null   str   
 2   source_job_id         3765 non-null   str   
 3   job_url               3765 non-null   str   
 4   title_raw             3765 non-null   str   
 5   company_name_raw      3765 non-null   str   
 6   company_url           3765 non-null   str   
 7   company_size_raw      3765 non-null   str   
 8   company_location_raw  3765 non-null   str   
 9   industry_raw          3765 non-null   str   
 10  salary_raw            3765 non-null   str   
 11  experience_raw        3765 non-null   str   
 12  education_raw         3765 non-null   str   
 13  location_raw          3765 non-null   str   
 14  work_mode_raw         3765 non-null   str   
 15  employmen

,Kiểu Dữ Liệu,Số lượng Null/None,"Số lượng Chuỗi Rỗng ("""")",Tổng Số Khuyết,Tỷ Lệ Khuyết (%)
preferred_skills_raw,str,0,3765,3765,100.00
updated_at_raw,str,0,2344,2344,62.26
overview_raw,str,0,2041,2041,54.21
company_size_raw,str,0,873,873,23.19
education_raw,str,0,669,669,17.77
working_time_raw,str,0,254,254,6.75
company_logo_url,str,0,31,31,0.82
work_mode_raw,str,0,22,22,0.58
requirements_raw,str,0,16,16,0.42
benefits_raw,str,0,3,3,0.08


--- 3. BẢNG KIỂM TRA TRÙNG LẶP (DUPLICATES REPORT) ---


,Tiêu Chí Trùng Lặp,Số Bản Ghi Trùng,Tỷ Lệ Trùng (%)
0,"Theo cặp khóa nguồn [source, source_job_id]",0,0.00
1,Theo đường dẫn [job_url],0,0.00
2,"Theo cùng vị trí & cùng công ty [title_raw, company_name...",139,3.69


## 4. Xem Mẫu Dữ Liệu Sau Khi Ghép (Toàn Bộ 39 Cột - Dễ Quan Sát)

Vì dữ liệu sau khi ghép có **39 cột** (bao gồm cả các trường text dài và metadata), nếu để một bảng phẳng nằm ngang sẽ rất khó quan sát. Chúng ta tổ chức hiển thị theo các cách sau để bạn dễ kiểm tra nhất:

1.  **Cách 1: Hiển thị Bảng Chuyển Vị (Transposed `.T`)**: Chuyển toàn bộ 39 cột thành danh sách dọc. Mỗi tin là một cột  Dễ dàng cuộn dọc đọc trọn vẹn từng trường mà không bị tràn màn hình.
2.  **Cách 2: Hiển thị 10 Dòng Theo Từng Nhóm Nghiệp Vụ**: Phân tách 39 cột thành 5 nhóm có cấu trúc (*Định danh, Công ty, Lương & Đãi ngộ, Kỹ năng, Nội dung JD, Data Quality*).
3.  **Cách 3: Bảng Phẳng Đầy Đủ (Toàn bộ 39 cột x 10 dòng)**: Mở rộng `display.max_columns = None` để bạn có thể cuộn ngang xem tất cả các cột nếu cần.

In [9]:
# --- CÁCH 1: HIỂN THỊ DẠNG CHUYỂN VỊ (TRANSPOSED VIEW) ---
# 39 cột sẽ hiển thị theo chiều dọc (Index), các mẫu tin hiển thị theo cột
print(' [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:')
sample_3 = df_merged.sample(3, random_state=42)
display(sample_3.T)


 [CÁCH 1] XEM DỌC TOÀN BỘ 39 CỘT CỦA 3 TIN TUYỂN DỤNG MẪU:


,1553,2989,220
detail_id,topcv_2319430,vietnamworks_2100743,topcv_2323134
source,topcv,vietnamworks,topcv
source_job_id,2319430,2100743,2323134
job_url,https://www.topcv.vn/brand/shinhanfinance1/tuyen-dung/cy...,https://www.vietnamworks.com/ky-su-quang-luong-tu--21007...,https://www.topcv.vn/viec-lam/nhan-vien-thiet-ke-do-hoa-...
title_raw,Cyber Security Supervisor,Kỹ Sư Quang Lượng Tử,Nhân Viên Thiết Kế Đồ Họa (Graphic Designer)
company_name_raw,SHINHAN FINANCE,Viện Nghiên Cứu Công Nghệ Chiến Lược Viettel (VIST),Công ty cổ phần Sphinx
company_url,https://www.topcv.vn/brand/shinhanfinance1?id=198515,https://www.vietnamworks.com/nha-tuyen-dung/vien-nghien-...,https://www.topcv.vn/cong-ty/cong-ty-co-phan-sphinx/6781...
company_size_raw,,,100-499 nhân viên
company_location_raw,"Pico Plaza, 20 Cộng Hòa, Phường Bảy Hiền, Hồ Chí Minh","Tòa nhà Viettel, số 380 Lạc Long Quân, quận Tây Hồ, Hà Nội","Tầng 1, Tòa nhà HL, Số 6/82 Đường Duy Tân, Phường Dịch V..."
industry_raw,"IT phần mềm, Công nghệ thông tin",Phần Mềm CNTT/Dịch vụ Phần mềm (Khoa Học & Kỹ Thuật > Kỹ...,"Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I..."


In [10]:
# --- CÁCH 2: XEM MẪU 10 DÒNG THEO TỪNG NHÓM NGHIỆP VỤ ---
sample_10 = df_merged.sample(10, random_state=42)

# Nhóm 1: Định danh & Truy vết nguồn (7 cột)
group_identity = ['source', 'source_job_id', 'job_url', 'crawled_at', 'raw_date', 'raw_file_name', 'listing_file_name']
print(' NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)')
display(sample_10[group_identity])

# Nhóm 2: Thông tin Công ty (7 cột)
group_company = ['company_name_raw', 'company_url', 'company_logo_url', 'company_size_raw', 'company_location_raw', 'industry_raw', 'location_raw']
print('\n NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)')
display(sample_10[group_company])

# Nhóm 3: Vị trí, Lương & Đãi ngộ tuyển dụng (8 cột)
group_job_specs = ['title_raw', 'salary_raw', 'experience_raw', 'education_raw', 'work_mode_raw', 'employment_type_raw', 'working_time_raw', 'deadline_raw']
print('\n NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)')
display(sample_10[group_job_specs])

# Nhóm 4: Kỹ năng & Ngày tháng (6 cột)
group_skills = ['skills_raw', 'required_skills_raw', 'preferred_skills_raw', 'posted_at_raw', 'published_at_raw', 'updated_at_raw']
print('\n NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)')
display(sample_10[group_skills])

# Nhóm 5: Nội dung chi tiết văn bản (4 cột)
group_texts = ['overview_raw', 'responsibilities_raw', 'requirements_raw', 'benefits_raw']
print('\n NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)')
display(sample_10[['title_raw'] + group_texts].map(lambda x: (str(x)[:80] + '...') if len(str(x)) > 80 else str(x)))


 NHÓM 1: ĐỊNH DANH & TRUY VẾT NGUỒN (7 CỘT)


,source,source_job_id,job_url,crawled_at,raw_date,raw_file_name,listing_file_name
1553,topcv,2319430,https://www.topcv.vn/brand/shinhanfinance1/tuyen-dung/cy...,2026-10-12T10:02:30.440307,2026-10-12,TCV001.json,page_001.json
2989,vietnamworks,2100743,https://www.vietnamworks.com/ky-su-quang-luong-tu--21007...,2026-10-16T21:36:56.351176,2026-10-16,VNW001.json,page_001.json
220,topcv,2323134,https://www.topcv.vn/viec-lam/nhan-vien-thiet-ke-do-hoa-...,2026-10-11T12:00:42.571603,2026-10-11,TCV006.json,page_001.json
2968,vietnamworks,2120312,https://www.vietnamworks.com/qa-game-tester-179213179158...,2026-10-16T21:36:47.875635,2026-10-16,VNW001.json,page_001.json
1971,topcv,2208886,https://www.topcv.vn/viec-lam/devops-engineer/2208886.html,2026-10-12T20:30:18.488111,2026-10-12,TCV003.json,page_003.json
1644,vietnamworks,2098486,https://www.vietnamworks.com/business-development-partne...,2026-10-12T09:50:18.095459,2026-10-12,VNW001.json,page_001.json
1979,topcv,2287707,https://www.topcv.vn/viec-lam/thuc-tap-sinh-kinh-doanh/2...,2026-10-12T20:32:10.445622,2026-10-12,TCV003.json,page_003.json
298,topcv,2318052,https://www.topcv.vn/viec-lam/thuc-tap-sinh-lap-trinh-de...,2026-10-11T12:15:47.550826,2026-10-11,TCV006.json,page_006.json
3336,topcv,2297419,https://www.topcv.vn/viec-lam/sa-manager-truong-phong-ho...,2026-10-18T19:21:32.044708,2026-10-18,TCV003.json,page_003.json
2440,vietnamworks,2118947,https://www.vietnamworks.com/data-analyst-tai-chinh-1791...,2026-10-13T19:51:55.118230,2026-10-13,VNW001.json,page_001.json



 NHÓM 2: THÔNG TIN DOANH NGHIỆP (7 CỘT)


,company_name_raw,company_url,company_logo_url,company_size_raw,company_location_raw,industry_raw,location_raw
1553,SHINHAN FINANCE,https://www.topcv.vn/brand/shinhanfinance1?id=198515,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,,"Pico Plaza, 20 Cộng Hòa, Phường Bảy Hiền, Hồ Chí Minh","IT phần mềm, Công nghệ thông tin","Pico Plaza, 20 Cộng Hòa, Phường Bảy Hiền, Hồ Chí Minh"
2989,Viện Nghiên Cứu Công Nghệ Chiến Lược Viettel (VIST),https://www.vietnamworks.com/nha-tuyen-dung/vien-nghien-...,https://images.vietnamworks.com/pictureofcompany/c0/1146...,,"Tòa nhà Viettel, số 380 Lạc Long Quân, quận Tây Hồ, Hà Nội",Phần Mềm CNTT/Dịch vụ Phần mềm (Khoa Học & Kỹ Thuật > Kỹ...,"Ha Noi, Ha Noi, Vietnam"
220,Công ty cổ phần Sphinx,https://www.topcv.vn/cong-ty/cong-ty-co-phan-sphinx/6781...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,100-499 nhân viên,"Tầng 1, Tòa nhà HL, Số 6/82 Đường Duy Tân, Phường Dịch V...","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...","Tòa nhà HL, số 6/82, Duy Tân, Dịch Vọng Hậu, Phường Cầu ..."
2968,"Nexon Networks Vina Co. Ltd,",https://www.vietnamworks.com/nha-tuyen-dung/nexon-networ...,https://images.vietnamworks.com/pictureofcompany/54/1122...,,"UOA TOWER, Tân Trào, Tân Phú, District 7, Thành phố Hồ C...",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn...,"UOA TOWER, Tân Trào, Tân Phú, District 7, Thành phố Hồ C..."
1971,CÔNG TY CỔ PHẦN THIẾT BỊ VÀ TRUYỀN THÔNG NGS,https://www.topcv.vn/cong-ty/cong-ty-co-phan-thiet-bi-va...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,500-1000 nhân viên,"Tòa 6Th Element Nguyễn Văn Huyên, Khu đô thị Tây Hồ Tây,...","IT phần mềm, Công nghệ thông tin","Tầng 4, sảnh M Tòa nhà 6th Element, Nguyễn Văn Huyên, Ng..."
1644,CÔNG TY CỔ PHẦN CÔNG NGHỆ TRÍ TUỆ NHÂN TẠO VBEE AITALK,https://www.vietnamworks.com/nha-tuyen-dung/cong-ty-co-p...,https://images.vietnamworks.com/pictureofcompany/d4/1147...,,"Số 35A ngõ 32 đường Bưởi, Phường Giảng Võ, TP Hà Nội, Vi...",Phần Mềm CNTT/Dịch vụ Phần mềm (Kinh Doanh > Bán Hàng/Ph...,"Tầng 15, tòa nhà Ngọc khánh Plaza, Số 1 Phạm Huy Thông, ..."
1979,Công ty cổ phần Công nghệ Windsoft Việt Nam,https://www.topcv.vn/cong-ty/cong-ty-co-phan-cong-nghe-w...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,"299 Trung Kính, Quận Cầu Giấy, Hà Nội","Kinh doanh / Bán hàng, IT phần mềm, Công nghệ thông tin","106 BT2A, Khu đô thị Mễ Trì, Phường Từ Liêm, Hà Nội"
298,CÔNG TY TNHH THƯƠNG MẠI KỸ THUẬT HỒNG THÁI,https://www.topcv.vn/cong-ty/cong-ty-tnhh-thuong-mai-ky-...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,25-99 nhân viên,"Số 58, Huỳnh Mẫn Đạt, phường 19, quận Bình Thạnh, TP.HCM","IT phần mềm, Công nghệ thông tin","Tòa nhà số 58 đường Huỳnh Mẫn Đạt, Phường Thạnh Mỹ Tây, ..."
3336,CÔNG TY TNHH BÁNH NGỌT CJ VIỆT NAM,https://www.topcv.vn/cong-ty/cong-ty-tnhh-banh-ngot-cj-v...,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,100-499 nhân viên,"Lầu 14 tòa nhà CJ, 2Bis-4-6 Lê Thánh Tôn, P. Bến Nghé, Q...","IT phần mềm, Công nghệ thông tin","Lầu 14, Tòa nhà CJ, số 2Bis-6 Lê Thánh Tôn, Phường Sài G..."
2440,Công ty Cổ phần Vàng bạc đá quý Bảo Tín Mạnh Hải,https://www.vietnamworks.com/nha-tuyen-dung/cong-ty-co-p...,https://images.vietnamworks.com/pictureofcompany/7b/1142...,,"Tầng 19 20 Tháp A, Tòa nhà HUD TOWER, số 37 Lê Văn Lương...",Bán lẻ/Bán sỉ (Công Nghệ Thông Tin/Viễn Thông > Data Eng...,"Hà Nội, Việt Nam"



 NHÓM 3: VỊ TRÍ, LƯƠNG & ĐIỀU KIỆN TUYỂN DỤNG (8 CỘT)


,title_raw,salary_raw,experience_raw,education_raw,work_mode_raw,employment_type_raw,working_time_raw,deadline_raw
1553,Cyber Security Supervisor,Thoả thuận,5 năm,Đại Học trở lên,Onsite,Quản lý / Giám sát,Toàn thời gian,2026-11-04T23:59:59+07:00
2989,Kỹ Sư Quang Lượng Tử,"Tới $ 2,500 /tháng",1 năm,,Onsite,Nhân viên,Toàn thời gian,2026-11-15T23:59:59+07:00
220,Nhân Viên Thiết Kế Đồ Họa (Graphic Designer),Thoả thuận,1 năm,Cao Đẳng trở lên,Onsite,Nhân viên,Toàn thời gian,2026-11-05T23:59:59+07:00
2968,QA Game Tester,Thương lượng,0 năm,,Onsite,Mới Tốt Nghiệp,Toàn thời gian,2026-11-15T23:59:59+07:00
1971,DevOps Engineer,30000000,4 năm,Đại Học trở lên,Onsite,Nhân viên,Toàn thời gian,2026-10-29T23:59:59+07:00
1644,Business Development Partnership - Ecosystem,13tr-20tr ₫/tháng,2 năm,,Onsite,Nhân viên,Toàn thời gian,2026-10-28T23:59:59+07:00
1979,Thực Tập Sinh Kinh Doanh,3000000,Không yêu cầu,Cao Đẳng trở lên,Onsite,Thực tập sinh,Thực tập,2026-10-29T23:59:59+07:00
298,Thực Tập Sinh Lập Trình (Developer Intern),Thoả thuận,Không yêu cầu,Đại Học trở lên,Onsite,Thực tập sinh,Thực tập,2026-10-31T23:59:59+07:00
3336,SA Manager / Trưởng Phòng Hỗ Trợ Kinh Doanh,Thoả thuận,3 năm,Đại Học trở lên,Onsite,Trưởng/Phó phòng,Toàn thời gian,2026-11-11T23:59:59+07:00
2440,Data Analyst (Tài Chính),Thương lượng,3 năm,Cử nhân,Onsite,Nhân viên,Toàn thời gian,2026-11-12T23:59:59+07:00



 NHÓM 4: KỸ NĂNG & MỐC THỜI GIAN (6 CỘT)


,skills_raw,required_skills_raw,preferred_skills_raw,posted_at_raw,published_at_raw,updated_at_raw
1553,"Không yêu cầu kinh nghiệm quản lý, 5 năm kinh nghiệm chu...","Không yêu cầu kinh nghiệm quản lý, 5 năm kinh nghiệm chu...",,2026-10-05,2026-10-05,
2989,"Kỹ Thuật Quang Học, Truyền Dẫn Quang, Quantum Mechanics,...","Kỹ Thuật Quang Học, Truyền Dẫn Quang, Quantum Mechanics,...",,2026-10-16T16:22:39+07:00,2026-10-16T16:22:39+07:00,2026-10-16T16:22:39+07:00
220,"1 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...","1 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...",,2026-10-06,2026-10-06,
2968,"Game Testing, Quality Assurance, Manual Testing, English...","Game Testing, Quality Assurance, Manual Testing, English...",,2026-10-16T13:23:11+07:00,2026-10-16T13:23:11+07:00,2026-10-16T13:23:11+07:00
1971,"4 năm kinh nghiệm chuyên môn, DevOps Engineer, IT - Phần...","4 năm kinh nghiệm chuyên môn, DevOps Engineer, IT - Phần...",,2026-09-29,2026-09-29,
1644,"Đàm Phán Thương Thuyết, Giao tiếp, Phát Triển Kinh Doanh...","Đàm Phán Thương Thuyết, Giao tiếp, Phát Triển Kinh Doanh...",,2026-09-28T15:02:36+07:00,2026-09-28T15:02:36+07:00,2026-09-30T17:30:55+07:00
1979,"Không yêu cầu kinh nghiệm chuyên môn, Kinh doanh phần mề...","Không yêu cầu kinh nghiệm chuyên môn, Kinh doanh phần mề...",,2026-09-29,2026-09-29,
298,"Không yêu cầu kinh nghiệm chuyên môn, Fullstack Develope...","Không yêu cầu kinh nghiệm chuyên môn, Fullstack Develope...",,2026-10-01,2026-10-01,
3336,"3 năm kinh nghiệm quản lý, 3 năm kinh nghiệm chuyên môn,...","3 năm kinh nghiệm quản lý, 3 năm kinh nghiệm chuyên môn,...",,2026-10-12,2026-10-12,
2440,"SQL, Tableau BI, Business Analysis, Phân Tích Dữ Liệu, F...","SQL, Tableau BI, Business Analysis, Phân Tích Dữ Liệu, F...",,2026-10-13T16:01:50+07:00,2026-10-13T16:01:50+07:00,2026-10-13T16:01:50+07:00



 NHÓM 5: NỘI DUNG VĂN BẢN CHI TIẾT (4 CỘT - TRÍCH ĐOẠN 80 KÝ TỰ)


,title_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw
1553,Cyber Security Supervisor,,The job holder needs to ensure that business operations ...,1. Qualification\n• Bachelor’s degree in Information Tec...,1. Colleagues:\nFriendly and dynamic working environment...
2989,Kỹ Sư Quang Lượng Tử,"Tập đoàn Công nghiệp – Viễn thông Quân đội (Viettel), th...",- Nghiên cứu và xây dựng kiến trúc các hệ thống phân phố...,- Có kinh nghiệm thực tế trong việc triển khai hệ thống ...,"Thưởng: Được hưởng các khoản quà, thưởng lễ tết, nghỉ m..."
220,Nhân Viên Thiết Kế Đồ Họa (Graphic Designer),,– Thiết kế ấn phẩm truyền thông:\n+ Thiết kế các ấn phẩm...,- Kinh nghiệm: Từ 01 – 02 năm ở vị trí Designer hoặc tươ...,1. Chế độ đãi ngộ:\n– Thu nhập: thoả thuận.\n– Phụ cấp o...
2968,QA Game Tester,"Nexon Networks VINA, responsible for the uppermost quali...",Chúng tôi tuyển dụng Nhân viên kiểm soát chất lượng để đ...,YÊU CẦU CÔNG VIỆC\n• Mới tốt nghiệp\n• Bằng cấp: Cao Đẳn...,"Thưởng: From 12 days Annual leave, 13th month salary\nCh..."
1971,DevOps Engineer,,"Deploy và vận hành microservices qua container hóa, CI/C...","Vận hành Linux production thành thạo (command line, syst...","Mức lương từ 30 - 45tr/tháng, lương tháng 13, thưởng kin..."
1644,Business Development Partnership - Ecosystem,,"•\tChủ động tìm kiếm, tiếp cận và phát triển khách hàng ...",2. Yêu cầu công việc\n2.1. Must-have\n•\tTừ 2 năm kinh n...,"Thưởng: Thưởng tháng 13, thưởng hiệu quả công việc, thưở..."
1979,Thực Tập Sinh Kinh Doanh,,Tiếp cận với tệp dữ liệu khách hàng có sẵn của công ty g...,"Sinh viên sắp ra trường, vừa ra trường\nCó thể đi làm fu...",Phụ cấp full-time 3.000.000đ\nTham gia các hoạt động các...
298,Thực Tập Sinh Lập Trình (Developer Intern),,Tham gia phát triển sản phẩm thực tế về AI Agent ở vị tr...,Bắt buộc\nĐã tốt nghiệp hoặc là sinh viên năm cuối ngành...,Phụ cấp thực tập: thỏa thuận theo năng lực khi phỏng vấn...
3336,SA Manager / Trưởng Phòng Hỗ Trợ Kinh Doanh,,Nhiệm vụ chính\nPhân tích và dự báo kinh doanh\nHỗ trợ v...,Năng lực & Kỹ năng Cá nhân đảm nhiệm vị trí cần có:\nBằn...,Lương tháng 13 | Review lương hàng năm | Thưởng KPI giữa...
2440,Data Analyst (Tài Chính),"Công ty Cổ phần Vàng Bạc Đá Quý Bảo Tín Mạnh Hải, với hơ...",Tại sao bạn nên gia nhập cùng chúng tôi\nCHẾ ĐỘ PHÚC LỢI...,Trình độ & kinh nghiệm\n- Tốt nghiệp Đại học trở lên các...,Thưởng: Thưởng tháng lương 13 cố định.\nThưởng các ngày ...


In [11]:
# --- CÁCH 3: BẢNG PHẲNG ĐẦY ĐỦ TOÀN BỘ 39 CỘT (SAMPLE 10 DÒNG) ---
print(' [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):')
display(sample_10)


 [CÁCH 3] BẢNG PHẲNG ĐẦY ĐỦ TẤT CẢ 39 CỘT (CÓ THỂ CUỘN NGANG ĐỂ XEM TOÀN BỘ):


,detail_id,source,source_job_id,job_url,title_raw,company_name_raw,company_url,company_size_raw,company_location_raw,industry_raw,salary_raw,experience_raw,education_raw,location_raw,work_mode_raw,employment_type_raw,working_time_raw,overview_raw,responsibilities_raw,requirements_raw,benefits_raw,required_skills_raw,preferred_skills_raw,published_at_raw,updated_at_raw,deadline_raw,crawled_at,raw_file_name,raw_date,dq_status,dq_errors,dq_checked_at,_dq_status,_dq_errors,_dq_checked_at,company_logo_url,skills_raw,posted_at_raw,listing_file_name
1553,topcv_2319430,topcv,2319430,https://www.topcv.vn/brand/shinhanfinance1/tuyen-dung/cy...,Cyber Security Supervisor,SHINHAN FINANCE,https://www.topcv.vn/brand/shinhanfinance1?id=198515,,"Pico Plaza, 20 Cộng Hòa, Phường Bảy Hiền, Hồ Chí Minh","IT phần mềm, Công nghệ thông tin",Thoả thuận,5 năm,Đại Học trở lên,"Pico Plaza, 20 Cộng Hòa, Phường Bảy Hiền, Hồ Chí Minh",Onsite,Quản lý / Giám sát,Toàn thời gian,,The job holder needs to ensure that business operations ...,1. Qualification\n• Bachelor’s degree in Information Tec...,1. Colleagues:\nFriendly and dynamic working environment...,"Không yêu cầu kinh nghiệm quản lý, 5 năm kinh nghiệm chu...",,2026-10-05,,2026-11-04T23:59:59+07:00,2026-10-12T10:02:30.440307,TCV001.json,2026-10-12,valid,[],2026-10-25T12:57:18.291970,valid,[],2026-10-25T12:57:18.291970,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"Không yêu cầu kinh nghiệm quản lý, 5 năm kinh nghiệm chu...",2026-10-05,page_001.json
2989,vietnamworks_2100743,vietnamworks,2100743,https://www.vietnamworks.com/ky-su-quang-luong-tu--21007...,Kỹ Sư Quang Lượng Tử,Viện Nghiên Cứu Công Nghệ Chiến Lược Viettel (VIST),https://www.vietnamworks.com/nha-tuyen-dung/vien-nghien-...,,"Tòa nhà Viettel, số 380 Lạc Long Quân, quận Tây Hồ, Hà Nội",Phần Mềm CNTT/Dịch vụ Phần mềm (Khoa Học & Kỹ Thuật > Kỹ...,"Tới $ 2,500 /tháng",1 năm,,"Ha Noi, Ha Noi, Vietnam",Onsite,Nhân viên,Toàn thời gian,"Tập đoàn Công nghiệp – Viễn thông Quân đội (Viettel), th...",- Nghiên cứu và xây dựng kiến trúc các hệ thống phân phố...,- Có kinh nghiệm thực tế trong việc triển khai hệ thống ...,"Thưởng: Được hưởng các khoản quà, thưởng lễ tết, nghỉ m...","Kỹ Thuật Quang Học, Truyền Dẫn Quang, Quantum Mechanics,...",,2026-10-16T16:22:39+07:00,2026-10-16T16:22:39+07:00,2026-11-15T23:59:59+07:00,2026-10-16T21:36:56.351176,VNW001.json,2026-10-16,valid,[],2026-10-25T12:57:19.725323,valid,[],2026-10-25T12:57:19.725323,https://images.vietnamworks.com/pictureofcompany/c0/1146...,"Kỹ Thuật Quang Học, Truyền Dẫn Quang, Quantum Mechanics,...",2026-10-16T16:22:39+07:00,page_001.json
220,topcv_2323134,topcv,2323134,https://www.topcv.vn/viec-lam/nhan-vien-thiet-ke-do-hoa-...,Nhân Viên Thiết Kế Đồ Họa (Graphic Designer),Công ty cổ phần Sphinx,https://www.topcv.vn/cong-ty/cong-ty-co-phan-sphinx/6781...,100-499 nhân viên,"Tầng 1, Tòa nhà HL, Số 6/82 Đường Duy Tân, Phường Dịch V...","Thiết kế đồ họa, Marketing / Truyền thông / Quảng cáo, I...",Thoả thuận,1 năm,Cao Đẳng trở lên,"Tòa nhà HL, số 6/82, Duy Tân, Dịch Vọng Hậu, Phường Cầu ...",Onsite,Nhân viên,Toàn thời gian,,– Thiết kế ấn phẩm truyền thông:\n+ Thiết kế các ấn phẩm...,- Kinh nghiệm: Từ 01 – 02 năm ở vị trí Designer hoặc tươ...,1. Chế độ đãi ngộ:\n– Thu nhập: thoả thuận.\n– Phụ cấp o...,"1 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...",,2026-10-06,,2026-11-05T23:59:59+07:00,2026-10-11T12:00:42.571603,TCV006.json,2026-10-11,valid,[],2026-10-25T12:57:18.283463,valid,[],2026-10-25T12:57:18.283463,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/...,"1 năm kinh nghiệm chuyên môn, Thiết kế đồ họa (Graphic D...",2026-10-06,page_001.json
2968,vietnamworks_2120312,vietnamworks,2120312,https://www.vietnamworks.com/qa-game-tester-179213179158...,QA Game Tester,"Nexon Networks Vina Co. Ltd,",https://www.vietnamworks.com/nha-tuyen-dung/nexon-networ...,,"UOA TOWER, Tân Trào, Tân Phú, District 7, Thành phố Hồ C...",Phần Mềm CNTT/Dịch vụ Phần mềm (Công Nghệ Thông Tin/Viễn